[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/03_multilingual_eval/03_multilingual_eval.ipynb)

# 03 · 多语言与跨语言评测（纯 numpy/pandas）

目标：把 **tokenizer fertility**、**分词税**、**上下文不公**、**chrF 翻译质量**、**公平聚合**、**跨语言一致性** 全部从零实现，并用 `assert` 验证。

路线：toy tokenizer 与 fertility → 跨语 fertility 比值表 → 分词税(成本) → 上下文不公 → chrF 从零 + 最差组 → ✏️ 练习(跨语一致 / fertility 偏置 / 覆盖 / 公平聚合) → 📖 答案 → 🧪 真实数据胶囊(真实 tiktoken + UDHR 文本)。

> 心智模型：**tokenizer 是一道对各语言收费不均的收费站。fertility = 每个词被切成几个 token，它一路下游变成成本、上下文、延迟的不公。**

## 1 · toy tokenizer 与 fertility

真实 tokenizer 是在语料上学合并表的（BPE/SentencePiece）。这里用一个**确定性的玩具 tokenizer** 抓住核心机制：
- 它有一个**学过的词表**（vocab，模拟在英语为主语料上学到的高频片段）；
- 切词时**贪心匹配**词表里的最长片段；匹配不到就**退回到单字符**（模拟 byte-fallback）。

于是词表覆盖好的语言 fertility 低，覆盖差的语言被切碎、fertility 高。**fertility = #tokens / #words**。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def greedy_tokenize(word, vocab):
    '''对单个 word 贪心匹配 vocab 里最长前缀；匹配不到则切单字符(byte-fallback)。返回 token 列表。'''
    toks, i = [], 0
    while i < len(word):
        best = None
        for j in range(len(word), i, -1):      # 从最长到最短找前缀
            if word[i:j] in vocab:
                best = word[i:j]; break
        if best is None:
            best = word[i]                      # 退回单字符
        toks.append(best); i += len(best)
    return toks

def tokenize_text(text, vocab):
    toks = []
    for w in text.split():
        toks.extend(greedy_tokenize(w, vocab))
    return toks

def fertility(text, vocab):
    n_words = max(len(text.split()), 1)
    return len(tokenize_text(text, vocab)) / n_words

# 一个偏向 English 的词表：英语常见整词/词缀在表里；其他语言片段大多不在
vocab = {'all','human','being','beings','are','born','free','and','equal','in','dignity','rights',
         'the','of','to','is','a','los','y','en','na','wa','ni',' to','ke','en'}
vocab |= {chr(c) for c in range(32,127)}     # 含 ASCII 单字符，保证英语总能切

en = 'all human beings are born free and equal in dignity and rights'
toks = tokenize_text(en, vocab)
f_en = fertility(en, vocab)
print('English tokens:', toks)
print(f'English fertility = {f_en:.2f}  (#tokens={len(toks)}, #words={len(en.split())})')
assert f_en < 1.3, '英语应被词表很好覆盖, fertility 接近 1'
print('✅ 词表覆盖好的语言 fertility 接近 1')

## 2 · 跨语言 fertility 比值表

现在对多种 “语言” 算 fertility。为了在纯玩具数据里**复现真实方向**（低资源/形态丰富语言 fertility 高），我们让非英语句子的词大多**不在词表**里、且更长（模拟屈折词缀与非拉丁退回字节）。

重点产出：**相对英语的 fertility 比值** —— 这就是分词税的源头。

In [ ]:
# 平行内容(同一句的不同语言近似), 故意让非英语词更长/更不在词表里
parallel = {
    'English':  'all human beings are born free and equal',
    'Spanish':  'todos seres humanos nacen libres iguales',
    'Swahili':  'watu wote wamezaliwa huru sawa katika',
    'Finnish':  'kaikki ihmiset syntyvaet vapaina tasavertaisina arvoltaan',
    'Hindi':    'sabhimanushyajanmasestvatantraevamsamaanhain',  # 模拟无空格/退回字节
}

rows = []
for lang, text in parallel.items():
    f = fertility(text, vocab)
    rows.append(dict(language=lang, words=len(text.split()),
                     tokens=len(tokenize_text(text, vocab)), fertility=round(f, 2)))
df = pd.DataFrame(rows).set_index('language')
df['rel_to_en'] = (df['fertility'] / df.loc['English', 'fertility']).round(2)
print(df)
assert df.loc['English', 'rel_to_en'] == 1.0
assert df.loc['Hindi', 'rel_to_en'] > 2.0, '非拉丁/无空格语言 fertility 应是英语数倍'
assert (df['rel_to_en'] >= 1.0).all(), '英语是 fertility 最低的(基准)'
print('✅ 跨语言 fertility 比值复现真实方向：英语最低，低资源语言数倍于它')

## 3 · 分词税：把 fertility 换算成成本

商用模型**按 token 计费**。同样语义，fertility 高的语言 token 多、付钱多。这就是 **token premium(分词税)**。

给定每语言要处理的**同义内容量**（用英语 token 数作基准）与单价，算各语言的实际花费与相对溢价。

In [ ]:
PRICE_PER_1K = 0.01      # $/1K tokens
BASE_EN_TOKENS = 1000    # 一段英语内容的 token 数(基准)

def token_cost(rel_fertility, base_en_tokens=BASE_EN_TOKENS, price=PRICE_PER_1K):
    '''同义内容在某语言的 token 数 = 基准 * 相对 fertility；返回 (tokens, cost_usd)。'''
    toks = base_en_tokens * rel_fertility
    return toks, toks / 1000 * price

cost_rows = []
for lang, rel in df['rel_to_en'].items():
    toks, cost = token_cost(rel)
    cost_rows.append(dict(language=lang, rel_fertility=rel,
                          tokens=int(toks), cost_usd=round(cost, 4),
                          premium=f'+{(rel-1)*100:.0f}%'))
cost_df = pd.DataFrame(cost_rows).set_index('language')
print(cost_df)
en_cost = cost_df.loc['English', 'cost_usd']
hi_cost = cost_df.loc['Hindi', 'cost_usd']
print(f'\n印地语为同义内容付 ${hi_cost:.4f} vs 英语 ${en_cost:.4f}  ->  {hi_cost/en_cost:.1f}x')
assert hi_cost > 2 * en_cost, 'fertility>2 的语言成本应是英语 2 倍以上'
print('✅ 分词税量化：母语决定你为相同内容付多少钱 —— 一种隐蔽的分配性危害')

## 4 · 上下文不公：同样窗口，不同容量

上下文窗口按 **token** 定。fertility 高的语言，同样的 token 预算塞进的**实际内容更少**。

可用内容(词) ≈ 上下文窗口(token) / fertility。给定 8192 token 窗口，看各语言能放多少词。

In [ ]:
CONTEXT_TOKENS = 8192

def usable_words(context_tokens, fert):
    return context_tokens / fert

ctx_rows = []
for lang, fert in df['fertility'].items():
    w = usable_words(CONTEXT_TOKENS, fert)
    ctx_rows.append(dict(language=lang, fertility=fert, usable_words=int(w)))
ctx_df = pd.DataFrame(ctx_rows).set_index('language')
ctx_df['rel_capacity'] = (ctx_df['usable_words'] / ctx_df.loc['English', 'usable_words']).round(2)
print(f'在 {CONTEXT_TOKENS}-token 窗口下各语言能放的词数:')
print(ctx_df)
assert ctx_df.loc['Hindi', 'usable_words'] < ctx_df.loc['English', 'usable_words'] / 2
assert abs(ctx_df.loc['English', 'rel_capacity'] - 1.0) < 1e-9
print('\n✅ 同一个上下文窗口, 低资源语言到手容量不到英语一半 —— 能力上限被母语决定')

## 5 · 翻译质量：从零实现 chrF + 最差组聚合

**chrF** = 字符级 n-gram 的 F 值（精确率与召回率的调和平均）。它无需分词、对形态丰富语言比 BLEU 更公平。

步骤：(1) 取候选/参考的字符 n-gram 多重集；(2) 算 n-gram 精确率 P 与召回率 R；(3) F_beta（chrF 常用 beta=2，偏重召回）。最后演示**最差组聚合**：报所有语言里 chrF 最低的那组，而非平均。

In [ ]:
from collections import Counter

def char_ngrams(text, n):
    t = text.replace(' ', '')               # chrF 通常忽略空格
    return Counter(t[i:i+n] for i in range(len(t)-n+1)) if len(t) >= n else Counter()

def chrf(candidate, reference, max_n=6, beta=2.0):
    '''字符级 n-gram F_beta, 对 n=1..max_n 取平均。返回 [0,1]。'''
    precs, recs = [], []
    for n in range(1, max_n + 1):
        c = char_ngrams(candidate, n)
        r = char_ngrams(reference, n)
        if not c or not r:
            continue
        overlap = sum((c & r).values())     # 多重集交集
        p = overlap / max(sum(c.values()), 1)
        rec = overlap / max(sum(r.values()), 1)
        precs.append(p); recs.append(rec)
    P = np.mean(precs) if precs else 0.0
    R = np.mean(recs) if recs else 0.0
    if P + R == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * P * R / (b2 * P + R)

# 完美匹配应得 1.0
assert abs(chrf('hello world', 'hello world') - 1.0) < 1e-9
# 部分匹配应在 (0,1)
partial = chrf('the cat sat', 'the cat ran')
print(f'完美匹配 chrF = 1.0 ; 部分匹配 chrF = {partial:.3f}')
assert 0.0 < partial < 1.0
# 完全不同应接近 0
assert chrf('abcdef', 'zzzzzz') < 0.1
print('✅ chrF 从零实现正确(完美=1, 部分介于 0~1, 无关≈0)')

In [ ]:
# 多语言译文质量 + 最差组聚合
translations = {
    'French':  ('le chat est noir', 'le chat est noir'),       # 完美
    'German':  ('die katze ist schwartz', 'die katze ist schwarz'),  # 仅拼写小错
    'Swahili': ('paka ni nyeusu', 'paka ni nyeusi'),           # 词尾差(形态)
    'Hindi':   ('billa kala hai', 'billi kaali hai'),          # 较多差异
}
qrows = [dict(language=l, chrf=round(chrf(cand, ref), 3))
         for l, (cand, ref) in translations.items()]
qdf = pd.DataFrame(qrows).set_index('language')
print(qdf)
macro = qdf['chrf'].mean()
worst = qdf['chrf'].min()
worst_lang = qdf['chrf'].idxmin()
print(f'\n宏平均 chrF = {macro:.3f}   最差组 = {worst:.3f} ({worst_lang})')
assert worst <= macro, '最差组必然 <= 平均'
assert worst_lang == 'Hindi'
print('✅ 平均分掩盖了最差语言 —— 永远把最差组和平均一起报')

---
## ✏️ 练习 1：fertility 偏置 —— 找出最贵的语言

给定一个 `{语言: 文本}` 字典与词表，实现 `fertility_bias(corpus, vocab, baseline)`：
返回一个 DataFrame，含每语言的 `fertility` 与相对 `baseline` 语言的 `rel`，并按 `rel` 降序。再返回 fertility 相对基准最高（最 “贵”）的语言名。

In [ ]:
def fertility_bias(corpus, vocab, baseline='English'):
    '''返回 (df 按 rel 降序, worst_lang)。df 索引为语言, 列含 fertility, rel。'''
    # TODO: 对每个语言算 fertility(text, vocab)；rel = fert / 基准语言 fert；
    #       构造 DataFrame, 按 rel 降序排序；worst_lang = rel 最大的语言
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
fdf, worst = fertility_bias(parallel, vocab, baseline='English')
assert list(fdf.index)[0] == worst, 'rel 降序后第一行应是最贵语言'
assert worst == 'Hindi', '本数据里 Hindi 最贵'
assert abs(fdf.loc['English', 'rel'] - 1.0) < 1e-9
assert (fdf['rel'] >= 1.0 - 1e-9).all(), '英语是基准, 其余 >= 1'
print(fdf)
print(f'✅ 练习 1 通过：最贵的语言是 {worst}')

## ✏️ 练习 2：语言覆盖度

一个产品只 “支持” 部分语言。给定**支持语言集** `supported` 与**用户语言分布** `user_dist`（`{语言: 用户占比}`，和为 1），实现 `coverage(supported, user_dist)`：
返回 `(语言种数覆盖率, 被服务人口占比)`。
- 语言种数覆盖率 = 支持的语言数 / 用户分布中出现的语言数
- 被服务人口占比 = 支持语言上的用户占比之和

In [ ]:
def coverage(supported, user_dist):
    # TODO: 返回 (n_supported / n_total_langs, 被服务人口占比)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
supported = {'English', 'Spanish', 'Chinese'}
user_dist = {'English': 0.40, 'Spanish': 0.15, 'Chinese': 0.20,
             'Hindi': 0.15, 'Swahili': 0.10}     # 后两者不被支持
lang_cov, pop_cov = coverage(supported, user_dist)
assert abs(lang_cov - 3/5) < 1e-9, '5 种语言里支持 3 种'
assert abs(pop_cov - 0.75) < 1e-9, '被服务人口 = 0.40+0.15+0.20'
print(f'语言种数覆盖率 = {lang_cov:.0%}  |  被服务人口占比 = {pop_cov:.0%}')
print('⚠️ 25% 的用户母语完全不被支持 —— 覆盖度审计揭示的盲区')
print('✅ 练习 2 通过')

## ✏️ 练习 3：公平聚合 macro / micro / worst-group

给定 `{语言: (分数, 样本数)}`，实现三种聚合：
- `macro`：各语言分数等权平均（每语言一票）
- `micro`：按样本数加权平均
- `worst`：最低分数

实现 `aggregate(scores)` 返回 `dict(macro=, micro=, worst=)`。

In [ ]:
def aggregate(scores):
    '''scores: {lang: (score, n_samples)}。返回 dict(macro, micro, worst)。'''
    # TODO: macro=分数均值; micro=sum(score*n)/sum(n); worst=min(分数)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 90 个样本的语言得 0.9, 10 个样本的语言得 0.2, 还有个中等
scores = {'big': (0.90, 900), 'mid': (0.60, 90), 'small': (0.20, 10)}
agg = aggregate(scores)
print({k: round(v, 3) for k, v in agg.items()})
assert abs(agg['macro'] - (0.9+0.6+0.2)/3) < 1e-9
assert abs(agg['micro'] - (0.9*900+0.6*90+0.2*10)/1000) < 1e-9
assert abs(agg['worst'] - 0.20) < 1e-9
assert agg['micro'] > agg['macro'] > agg['worst'], '大语言主导 micro, 最差组最低'
print('✅ 练习 3 通过：micro 被大语言抬高, worst 暴露被抛弃的语言')

## ✏️ 练习 4：跨语言一致性

把同一组问题用多种语言问模型，得到答案矩阵 `answers`（形状 `[n_questions, n_langs]`，每格是答案标签 id）。
实现 `consistency(answers)`：对每个问题，看各语言答案是否**全部一致**；返回**一致问题的比例**。
再实现 `per_question_agreement(answers)`：返回每个问题的 “多数答案占比”（众数计数 / 语言数）。

In [ ]:
def consistency(answers):
    # TODO: 对每行(问题)判断是否所有语言答案相同; 返回一致行的比例
    raise NotImplementedError

def per_question_agreement(answers):
    # TODO: 每行返回 (该行众数出现次数 / 列数)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 3 个问题 x 4 种语言。Q0 全一致, Q1 一个语言变卦, Q2 分裂
answers = np.array([
    [1, 1, 1, 1],   # 全一致
    [0, 0, 0, 1],   # 3:1
    [2, 2, 3, 3],   # 2:2
])
cons = consistency(answers)
agree = per_question_agreement(answers)
assert abs(cons - 1/3) < 1e-9, '只有 Q0 全一致'
assert np.allclose(agree, [1.0, 0.75, 0.5])
print(f'全一致问题比例 = {cons:.2f}  |  各问题多数占比 = {agree}')
print('✅ 练习 4 通过：跨语言一致率 + 逐题一致度量化了 “模型会不会因语言变卦”')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fertility_bias(corpus, vocab, baseline='English'):
    base = fertility(corpus[baseline], vocab)
    rows = []
    for lang, text in corpus.items():
        f = fertility(text, vocab)
        rows.append(dict(language=lang, fertility=round(f, 3), rel=round(f / base, 3)))
    out = pd.DataFrame(rows).set_index('language').sort_values('rel', ascending=False)
    worst_lang = out.index[0]
    return out, worst_lang

In [ ]:
# 练习 2 参考答案
def coverage(supported, user_dist):
    langs = set(user_dist)
    lang_cov = len(set(supported) & langs) / len(langs)
    pop_cov = sum(p for l, p in user_dist.items() if l in supported)
    return lang_cov, pop_cov

In [ ]:
# 练习 3 参考答案
def aggregate(scores):
    vals = [s for s, _ in scores.values()]
    macro = float(np.mean(vals))
    tot = sum(n for _, n in scores.values())
    micro = sum(s * n for s, n in scores.values()) / tot
    worst = float(min(vals))
    return dict(macro=macro, micro=micro, worst=worst)

In [ ]:
# 练习 4 参考答案
def consistency(answers):
    answers = np.asarray(answers)
    same = [len(set(row)) == 1 for row in answers]
    return float(np.mean(same))

def per_question_agreement(answers):
    answers = np.asarray(answers)
    out = []
    for row in answers:
        vals, counts = np.unique(row, return_counts=True)
        out.append(counts.max() / len(row))
    return np.array(out)

---
## 🧪 真实数据胶囊：真实 tokenizer + 真实多语文本

前面用玩具 tokenizer 讲清了机制。现在用**真实的 GPT tokenizer**（`tiktoken` 的 `cl100k_base`，即 GPT-3.5/4 所用）在**真实的多语言文本**（世界人权宣言第 1 条，公共领域）上测 fertility——复现 Ahia/Petrov 论文里的真实不公。

**三重回退**：① 优先用 `tiktoken`；② 若未安装/失败，回退到 **UTF-8 字节代理**（`tokens ≈ len(text.encode('utf-8'))`，非拉丁文字 UTF-8 字节天然更多，是真实成本的下界），结论方向一致。UDHR 文本为**内置真实字符串**，无需联网。

In [ ]:
# 真实 UDHR 第 1 条(公共领域), 6 种语言 —— 内置真实文本
UDHR = {
    'English': 'All human beings are born free and equal in dignity and rights.',
    'Spanish': 'Todos los seres humanos nacen libres e iguales en dignidad y derechos.',
    'Chinese': '人人生而自由，在尊严和权利上一律平等。',
    'Hindi':   'सभी मनुष्य जन्म से स्वतंत्र और समान हैं।',
    'Swahili': 'Watu wote wamezaliwa huru na sawa katika utu na haki.',
    'Finnish': 'Kaikki ihmiset syntyvät vapaina ja tasavertaisina arvoltaan.',
}

def real_token_counter():
    '''返回 (count_fn, source)。优先 tiktoken, 失败回退 UTF-8 字节代理。'''
    try:
        import tiktoken
        enc = tiktoken.get_encoding('cl100k_base')
        return (lambda s: len(enc.encode(s))), 'tiktoken cl100k_base (real GPT tokenizer)'
    except Exception as e:
        print('tiktoken 不可用, 回退 UTF-8 字节代理:', type(e).__name__)
        return (lambda s: len(s.encode('utf-8'))), 'UTF-8 byte proxy (fallback)'

count_tokens, src = real_token_counter()
print('token 计数来源:', src)

rows = []
for lang, text in UDHR.items():
    toks = count_tokens(text)
    chars = len(text)
    rows.append(dict(language=lang, chars=chars, tokens=toks,
                     tok_per_char=round(toks / chars, 3)))
real_df = pd.DataFrame(rows).set_index('language')
real_df['rel_tok_per_char'] = (real_df['tok_per_char'] /
                               real_df.loc['English', 'tok_per_char']).round(2)
print(real_df)
# 非拉丁文字(中/印地)每字符 token 数应显著高于英语 —— 真实的分词不公
assert real_df.loc['Chinese', 'tok_per_char'] > real_df.loc['English', 'tok_per_char']
assert real_df.loc['Hindi', 'rel_tok_per_char'] > 1.5
print('\n✅ 真实 tokenizer 证实：非拉丁文字每字符消耗更多 token —— Ahia/Petrov 测到的真实分词税')

**🧪 胶囊练习**：实现 `worst_paying_language(real_df)`：返回 `rel_tok_per_char` 最高（相对英语最 “贵”）的语言，及其相对英语的倍数。用它确认上面真实数据里最吃亏的语言。

In [ ]:
def worst_paying_language(real_df):
    # TODO: 返回 (rel_tok_per_char 最大的语言名, 其 rel 值)
    raise NotImplementedError

In [ ]:
# 自测
lang, mult = worst_paying_language(real_df)
assert mult >= real_df['rel_tok_per_char'].max() - 1e-9
assert mult > 1.0, '最贵语言相对英语应 > 1'
print(f'最吃亏的语言: {lang}, 每字符 token 数是英语的 {mult:.2f}x')
print('✅ 胶囊练习通过：真实数据里的分词税受害者已定位')

In [ ]:
# 📖 胶囊参考答案
def worst_paying_language(real_df):
    s = real_df['rel_tok_per_char']
    lang = s.idxmax()
    return lang, float(s.max())

### 小结
- **fertility = #tokens / #words**：tokenizer 对各语言切分效率不均，是一切跨语成本/上下文不公的源头。
- **分词税**：fertility 高的语言为相同语义付更多钱(可达英语数倍) —— 隐蔽的**分配性危害**。
- **上下文不公**：同样 token 窗口, 低资源语言到手内容不到英语一半 —— 能力上限被母语决定。
- **chrF** 字符级、对形态丰富语言比 BLEU 公平；指标的选择本身是价值取向。
- **公平聚合**：永远把 **worst-group + 宏平均 + 覆盖度** 一起报，一个总平均分会掩盖低资源语言的崩盘。
- **跨语言一致性**：同一问题不同语言会变卦 —— 既是校准漂移, 也是安全护栏的越狱攻击面。

下一站：**模块 04 · 记忆化与隐私** —— 模型逐字背下训练数据, 外人能不能问出来。